# Project Milestone - Analyzing Utah State Legislative Trends
**Team Info**

Gabriella Grover, u1387821@utah.edu, u1387821

Lucy Ballou, u199709@utah.edu, u1399709

## Data Acquisition
While our variables stayed the same, we pivoted our collection methods slightly. We originally planned to scrape the Utah Legislature website, but as we got started, it became clear that the content we needed wasn't being loaded directly in raw HTML. Instead, it was being loaded via JavaScript after the HTML had already fully loaded, so it was never reached by BeautifulSoup. On the bright side, this hinted at the fact that the data was being loaded via API, and we were able to find and get access to the JSON files. Therefore, after examining the terms and documentation and obtaining our token, we decided to use the datafiles on the https://le.utah.gov/data/developer.htm website instead of scraping.
Additionally, where we planned to use ACS data, we have instead decided to use Federal Reserve Economic Data (FRED) to provide some economic context for our legislative analysis. While exploring ACS data, we figured we didn't need our variables to be as granular as ACS offered. FRED data relies somewhat on ACS data to create their datasets, but they've already done the work of aggregating the data to output one overall average value per year, grouped by state, which is exactly what we needed.

## Federal Reserve Economic Data
Reference the README for the variables.

## Utah Legislative Data
**DO NOT RUN THE LEGISLATIVE CODE.** It will take hours and we have redacted the API token.

In [2]:
# importing packages
import requests
import pandas as pd
import json
import time

In [ ]:
# pulling data from the legislative API

TOKEN = "redacted"
BASE_URL = "https://glen.le.utah.gov/bills"

# defining sessions (pre-2016 is unavail with API)
sessions = [
    "2016GS", "2017GS", "2018GS", "2019GS", "2020GS",
    "2021GS", "2022GS", "2023GS", "2024GS", "2025GS", "2026GS"
]

# extracting bill numbers
all_bills = {}

for session in sessions:
    url = f"{BASE_URL}/{session}/billlist/{TOKEN}"
    response = requests.get(url)

    if response.status_code == 200:
        try:
            data = response.json()

            # handling the fact that some years are formatted as plain lists, some are objects, some are strings
            if isinstance(data, list):
                bill_list = data
            elif isinstance(data, dict) and "bills" in data:
                bill_list = data["bills"]
            else:
                print(f"{session}: Unexpected format")
                continue

            bill_numbers = [bill['number'] for bill in bill_list]
            all_bills[session] = bill_numbers
            print(f"{session}: {len(bill_numbers)} bills")
        except Exception as e:
            print(f"{session}: Failed - {e}")
    else:
        print(f"{session}: ERROR {response.status_code}")

    time.sleep(1)

total = sum(len(bills) for bills in all_bills.values())
print(f"\nTotal bills across sessions: {total}")

# actual extracting
def extract_bill_data(session, bill_number, token):
    url = f"{BASE_URL}/{session}/{bill_number}/{token}"
    response = requests.get(url)

    if response.status_code != 200:
        return None

    try:
        data = response.json()
    except:
        return None

    if isinstance(data, dict) and "bill" in data and isinstance(data["bill"], dict):
        data = data["bill"]

    if isinstance(data, str):
        try:
            data = json.loads(data)
        except:
            return None

    # skip if still not a dict
    if not isinstance(data, dict):
        return None

    is_old_format = "shorttitle" in data

    if is_old_format:
        # old format
        subjects = data.get("subjects", [])

        # committees from agendas
        agendas = data.get("agendas", [])
        committees = list(set(
            a.get("committee", a.get("committeeName", ""))
            for a in agendas
        )) if agendas else []

        # originating chamber from bill number prefix
        bill_id = data.get("bill", "")
        if bill_id.startswith("HB") or bill_id.startswith("HJ"):
            orig_chamber = "House"
        elif bill_id.startswith("SB") or bill_id.startswith("SJ"):
            orig_chamber = "Senate"
        else:
            orig_chamber = ""

        return {
            "bill_id": bill_id,
            "short_title": data.get("shorttitle", ""),
            "sponsor": data.get("sponsor", ""),
            "co_sponsors": "None",
            "floor_sponsor": data.get("floorsponsor", ""),
            "last_action": data.get("lastaction", ""),
            "last_action_owner": data.get("lastactionowner", ""),
            "subjects": "; ".join(subjects) if isinstance(subjects, list) else str(subjects),
            "num_substitutes": 0,
            "substitute_sponsors": "None",
            "active_version": data.get("version", ""),
            "subjects_changed": False,
            "committees": "; ".join(committees),
            "money_appropriated": data.get("monies", ""),
            "originating_chamber": orig_chamber,
            "final_chamber": "",
            "sponsor_id": data.get("sponsor", ""),
            "session_year": session[:4],
            "session_id": session,
        }

    else:
        # new format
        versions = data.get("billVersionList", [])

        # finding and extracting active version for co-sponsors and subjects
        active_version = None
        active_version_label = ""
        for version in versions:
            if version.get("activeVersion", False):
                active_version = version
                active_version_label = version.get("billNumber", "")
                break
        if active_version is None and versions:
            active_version = versions[-1]
            active_version_label = versions[-1].get("billNumber", "")

        subjects = []
        if active_version:
            subjects = list(set(
                s["description"] for s in active_version.get("subjectList", [])
            ))

        co_sponsors = []
        if active_version:
            co_sponsors = [
                cs.get("name", "") for cs in active_version.get("coSponsorList", [])
            ]

        # substitute stuff
        substitutes = [v for v in versions if int(v.get("subVersion", 0)) > 0]
        num_substitutes = len(substitutes)

        sub_sponsors = "; ".join(
            f"{v['billNumber']}: {v.get('versionSponsorName', 'Unknown')}"
            for v in substitutes
        )

        # whether subjects changed across versions?
        def get_subjects_set(version):
            return set(s["description"] for s in version.get("subjectList", []))

        if len(versions) > 1:
            original_subjects = get_subjects_set(versions[0])
            subjects_changed = any(
                get_subjects_set(v) != original_subjects for v in versions[1:]
            )
        else:
            subjects_changed = False

        # getting committees from agendaList
        committees = list(set(
            a["committeeName"] for a in data.get("agendaList", [])
        ))

        # finding original and final chamber from action history
        originating_chamber = data.get("primeSponsorHouse", "")
        final_chamber = ""
        for action in reversed(data.get("actionHistoryList", [])):
            if action.get("actionClass") in ("H", "S"):
                final_chamber = "House" if action["actionClass"] == "H" else "Senate"
                break

        return {
            "bill_id": data.get("billNumber", ""),
            "short_title": data.get("shortTitle", ""),
            "sponsor": data.get("primeSponsorName", ""),
            "co_sponsors": "; ".join(co_sponsors) if co_sponsors else "None",
            "floor_sponsor": data.get("floorSponsorName", ""),
            "last_action": data.get("lastAction", ""),
            "last_action_owner": data.get("lastActionOwner", ""),
            "subjects": "; ".join(subjects),
            "num_substitutes": num_substitutes,
            "substitute_sponsors": sub_sponsors if sub_sponsors else "None",
            "active_version": active_version_label,
            "subjects_changed": subjects_changed,
            "committees": "; ".join(committees),
            "money_appropriated": data.get("moniesAppropriated", ""),
            "originating_chamber": "House" if originating_chamber == "H" else "Senate",
            "final_chamber": final_chamber,
            "sponsor_id": data.get("primeSponsor", ""),
            "session_year": data.get("year", ""),
            "session_id": data.get("sessionID", ""),
        }


# extraction loop
all_rows = []

for session, bill_numbers in all_bills.items():
    print(f"\n--- {session}: {len(bill_numbers)} bills ---")

    for i, bill_number in enumerate(bill_numbers):
        result = extract_bill_data(session, bill_number, TOKEN)

        if result:
            all_rows.append(result)
        else:
            print(f"  Failed: {bill_number}")

        # progress update every 50 bills
        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1}/{len(bill_numbers)}")

        time.sleep(1)

    # save progress after each session
    df_progress = pd.DataFrame(all_rows)
    df_progress.to_csv("utah_bills_progress.csv", index=False)
    print(f"  Saved progress: {len(all_rows)} total rows so far")

# build dataframe
df = pd.DataFrame(all_rows)
df.to_csv("utah_bills_final.csv", index=False)
print(f"\nFinal dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())


2016GS: 821 bills
2017GS: 812 bills
2018GS: 817 bills
2019GS: 844 bills
2020GS: 834 bills
2021GS: 768 bills
2022GS: 814 bills
2023GS: 929 bills
2024GS: 934 bills
2025GS: 959 bills
2026GS: 1016 bills

Total bills across sessions: 9548

--- 2016GS: 821 bills ---
  Failed: HB0038
  Processed 50/821
  Failed: HB0094
  Processed 100/821
  Processed 150/821
  Processed 200/821
  Processed 250/821
  Failed: HB0301
  Processed 300/821
  Processed 350/821
  Processed 400/821
  Processed 450/821
  Processed 500/821
  Failed: SB0014
  Processed 550/821
  Processed 600/821
  Processed 650/821
  Processed 700/821
  Processed 750/821
  Processed 800/821
  Saved progress: 817 total rows so far

--- 2017GS: 812 bills ---
  Processed 50/812
  Processed 100/812
  Processed 150/812
  Failed: HB0168
  Processed 200/812
  Processed 250/812
  Failed: HB0257
  Processed 300/812
  Failed: HB0308
  Processed 350/812
  Failed: HB0393
  Processed 400/812
  Failed: HB0418
  Processed 450/812
  Processed 500/812
 

## Data Cleaning

There was little we wanted to clean with the ACS/FRED data, since most of the data came in the form we wanted it in (floats or integers). All monetary numbers were already inflation-adjusted as well.

For the legislative data, there are a couple of variables we need to clean. First, we need to pull the monetary amounts listed in "money_appropriated" as seperate variables (i.e. "education_fund"), then sum them in another column for "total_approrpriated". We also want to convert "subjects", "committees", "originating_chamber", "final_chamber", and most sponsor variables into binary variables for our logit regression. This will siginificanlty expand our variable counts, so we will probably make a new copy of the "ut_bills" dataframe specifically for the regression. We also will use the values in the "last_action" column to determine if a bill passed and was signed into law, then make new variables from those (e.g. "Governor Signed" = passed). There will be a single "Pass/Fail" variable and then a "Bill Location" variable that will be an ordinal range where 0 is the starting point (usually "Chamber/Filed" or received note from LFA) and the final value being "Passed". There are different categories/values in the "last_action" that mean the same location, which we will code for.  

In [16]:
# inspecting FREDdata.csv

fred = pd.read_csv('FREDdata.csv')
print(fred.head())
fred.dtypes

   observation_date  bach_or_higher  umemployment_rate  med_hh_income  \
0              2010            29.3                8.3        56700.0   
1              2011            29.7                7.2        55490.0   
2              2012            30.7                4.8        58340.0   
3              2013            31.3                4.2        61050.0   
4              2014            31.1                3.6        63380.0   

   poverty_rate  homeownership_rate       gdp  resident_population  real_gdp  \
0          13.3                72.5  118296.3             2775.413  133739.1   
1          13.6                71.4  125137.8             2814.797  138301.1   
2          13.0                71.1  129431.4             2854.146  140622.5   
3          12.6                70.9  135344.3             2898.773  145026.9   
4          11.8                70.9  142627.7             2938.327  150076.3   

   rental_vacancy_rate    lfp  
0                  7.2  68.83  
1               

observation_date         int64
bach_or_higher         float64
umemployment_rate      float64
med_hh_income          float64
poverty_rate           float64
homeownership_rate     float64
gdp                    float64
resident_population    float64
real_gdp               float64
rental_vacancy_rate    float64
lfp                    float64
dtype: object

In [11]:
# inspecting utah_bills_final.csv

ut_bills = pd.read_csv('utah_bills_final.csv')
ut_bills.head()

,bill_id,short_title,sponsor,co_sponsors,floor_sponsor,last_action,last_action_owner,subjects,num_substitutes,substitute_sponsors,active_version,subjects_changed,committees,money_appropriated,originating_chamber,final_chamber,sponsor_id,session_year,session_id
0,HB0001,Public Education Base Budget Amendments,ELIASS,NaN,STEPHHA,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Public Education; Appropriations,0,NaN,HB0001S01,False,NaN,"This bill appropriates $500,000 in operating a...",House,NaN,ELIASS,2016,2016GS
1,HB0002,New Fiscal Year Supplemental Appropriations Act,SANPED,NaN,HILLYLW,Governor Line Item Veto,Lieutenant Governor's office for filing,Budgeting; Appropriations,0,NaN,HB0002,False,NaN,"This bill appropriates $540,382,500 in operati...",House,NaN,SANPED,2016,2016GS
2,HB0003,Appropriations Adjustments,SANPED,NaN,HILLYLW,Governor Line Item Veto,Lieutenant Governor's office for filing,Budgeting; Appropriations,0,NaN,HB0003,False,NaN,"This bill appropriates ($5,417,500) in operati...",House,NaN,SANPED,2016,2016GS
3,HB0005,"Natural Resources, Agriculture, and Environmen...",MCKELMK,NaN,HINKIDP,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Natural Resources; Appropriations,0,NaN,HB0005,False,NaN,"This bill appropriates $320,321,400 in operati...",House,NaN,MCKELMK,2016,2016GS
4,HB0006,Executive Offices and Criminal Justice Base Bu...,HUTCHEK,NaN,THATCDW,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Law Enforcement and Criminal Justic...,0,NaN,HB0006S01,False,NaN,"This bill appropriates ($618,200) in operating...",House,NaN,HUTCHEK,2016,2016GS


In [45]:
print(ut_bills['committees'].unique())
print()
print(ut_bills['final_chamber'].unique())
print()
print(len(ut_bills['last_action'].unique()), ut_bills['last_action'].unique())

[nan 'House Government Operations Committee'
 'Senate Government Operations and Political Subdivisions Committee; House Government Operations Committee'
 'Senate Government Operations and Political Subdivisions Committee'
 'Senate Business and Labor Committee'
 'Senate Judiciary, Law Enforcement, and Criminal Justice Committee'
 'House Retirement and Independent Entities Committee; Senate Retirement and Independent Entities Committee'
 'House Law Enforcement and Criminal Justice Committee; Senate Judiciary, Law Enforcement, and Criminal Justice Committee'
 'Senate Government Operations and Political Subdivisions Committee; House Political Subdivisions Committee'
 'House Health and Human Services Committee; Senate Health and Human Services Committee'
 'Senate Natural Resources, Agriculture, and Environment Committee'
 'Senate Judiciary, Law Enforcement, and Criminal Justice Committee; House Judiciary Committee'
 'Senate Revenue and Taxation Committee'
 'Senate Revenue and Taxation Commi

In [46]:
# converting 'last_action' to 'passed' binary variable
ut_bills['passed'] = ut_bills['last_action'].isin(['Governor Signed','Became Law w/o Governor Signature']).astype(int)

# converting 'last_action' to ordinal range 'final_location'
ut_bills_bi = ut_bills

location0 = ['House/ filed', 'Senate/ filed', '']
location1 = []
location2 = []
location3 = []

ut_bills_bi = ut_bills_bi['last_action']

ut_bills.head()

,bill_id,short_title,sponsor,co_sponsors,floor_sponsor,last_action,last_action_owner,subjects,num_substitutes,substitute_sponsors,active_version,subjects_changed,committees,money_appropriated,originating_chamber,final_chamber,sponsor_id,session_year,session_id,passed
0,HB0001,Public Education Base Budget Amendments,ELIASS,NaN,STEPHHA,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Public Education; Appropriations,0,NaN,HB0001S01,False,NaN,"This bill appropriates $500,000 in operating a...",House,NaN,ELIASS,2016,2016GS,1
1,HB0002,New Fiscal Year Supplemental Appropriations Act,SANPED,NaN,HILLYLW,Governor Line Item Veto,Lieutenant Governor's office for filing,Budgeting; Appropriations,0,NaN,HB0002,False,NaN,"This bill appropriates $540,382,500 in operati...",House,NaN,SANPED,2016,2016GS,0
2,HB0003,Appropriations Adjustments,SANPED,NaN,HILLYLW,Governor Line Item Veto,Lieutenant Governor's office for filing,Budgeting; Appropriations,0,NaN,HB0003,False,NaN,"This bill appropriates ($5,417,500) in operati...",House,NaN,SANPED,2016,2016GS,0
3,HB0005,"Natural Resources, Agriculture, and Environmen...",MCKELMK,NaN,HINKIDP,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Natural Resources; Appropriations,0,NaN,HB0005,False,NaN,"This bill appropriates $320,321,400 in operati...",House,NaN,MCKELMK,2016,2016GS,1
4,HB0006,Executive Offices and Criminal Justice Base Bu...,HUTCHEK,NaN,THATCDW,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Law Enforcement and Criminal Justic...,0,NaN,HB0006S01,False,NaN,"This bill appropriates ($618,200) in operating...",House,NaN,HUTCHEK,2016,2016GS,1


## Final Analysis

We are still planning to do a logistical regression to predict the passage of a bill based on economic & demographic indicators. Once we clean the data to be binary-coded, we will run an exploratory analysis on the general FRED data to determine the correlated variables. We will then build the model, analyze the coefficients to determine the variable relationships, and try cross-validating to determine if there is room for improvement.

We will also use a classifier to determine bill success based on different bill aspects. We are leaning towards decision trees/random forests for the ease of presentation and it's capability of handling non-numerical variables. We would use the divided variables of "last_action", "subjects", ect. to determine what categories are the most influential on a bills passage.

## Moving Forward

As stated above, our next step is to clean, organize, and merge our data before beginning our analysis. We message frequently about our project progress and delegate tasks well, and will continue to do so throughout these next steps. We will try to meet in person about once per week to check in, troubleshoot, and plan. We have our analysis methods planned out, but are prepared to make collaborative decisions about switching our methods if the analysis isn't going to plan.

